In [37]:
import os
import certifi
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain import hub
from langchain.tools import tool
import requests


In [38]:
from langchain.agents import create_react_agent, AgentExecutor

In [39]:
from pathlib import Path

os.environ["SSL_CERT_FILE"] = certifi.where()
env_path = Path(".env")
if not env_path.is_file():
    env_path = Path("research/.env")
load_dotenv(env_path)

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
WEATHERSTACK_API_KEY = os.getenv("WEATHERSTACK_API_KEY")

In [40]:
search_tool = TavilySearchResults(max_results=3)

In [ ]:
@tool
def get_weather_data(city: str) -> str:
    """
    Fetch current weather information for a city.
    """

    url = (
        f"https://api.weatherstack.com/current?"
        f"access_key={WEATHERSTACK_API_KEY}&query={city}"
    )

    response = requests.get(url)

    data = response.json()

    if "current" not in data:
        return f"Could not fetch weather data for {city}"

    return (
        f"City: {city}\n"
        f"Temperature: {data['current']['temperature']}°C\n"
        f"Weather: {data['current']['weather_descriptions'][0]}\n"
        f"Humidity: {data['current']['humidity']}%"
    )

In [42]:
result = search_tool.invoke("Give me latest news on AI")
result

[{'url': 'https://www.artificialintelligence-news.com',
  'content': '1159 view(s)\n\nSee all\n\n## Latest\n\nView All Latest\n\nGovernance, Regulation & Policy\n\nNovember 3, 2026\n\n# House bill revives OTA as AI reshapes legislative expertise\n\nGovernance, Regulation & Policy\n\nNovember 2, 2026\n\n# India’s DEPA: Privacy-preserving AI training data\n\nFinance AI\n\nOctober 30, 2026\n\n# Banks block AI agents as US open banking rule stalls\n\nAI in Action\n\nOctober 28, 2026\n\n# UnitedHealthcare expands AI assistant following reported 90% resolution rate\n\nGovernance, Regulation & Policy\n\nOctober 26, 2026\n\n# Argyll warns UK AI held back by grid queue as Microsoft exits Nscale site\n\nCybersecurity AI\n\nOctober 23, 2026\n\n# OpenAI and Poland’s NASK partner on AI safety research\n\nAI in Action\n\nOctober 23, 2026\n\n# Microsoft AI diagnostic tool outperforms doctors in clinical tests [...] January 31, 2024\n\n#### Machine Learning\n\n### Gartner outlines four AI tiers in war

In [43]:
#LLM
llm = ChatOpenAI(model="gpt-4o", temperature=0, openai_api_key=OPENAI_API_KEY)

In [44]:
response = llm.invoke("What year is it?")
response

AIMessage(content='As of my last update, it is 2023.', response_metadata={'token_usage': {'completion_tokens': 12, 'prompt_tokens': 12, 'total_tokens': 24, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o', 'system_fingerprint': 'fp_e665f1271d', 'finish_reason': 'stop', 'logprobs': None}, id='run-9c81d729-73b4-47c4-a0bf-ccca4bd62f5d-0')

In [45]:
prompt = hub.pull("hwchase17/react")

/Users/sandeepchaudhary/anaconda3/envs/langagent/lib/python3.11/site-packages/langchain/hub.py:86: DeprecationWarning: The `langchainhub sdk` is deprecated.
Please use the `langsmith sdk` instead:
  pip install langsmith
Use the `pull_prompt` method.
  res_dict = client.pull_repo(owner_repo_commit)


In [46]:
tools = [search_tool, get_weather_data]

In [47]:
agent = create_react_agent(llm=llm, tools=tools, prompt=prompt)

In [48]:
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    handle_parsing_errors=True,
)

In [50]:
response = agent_executor.invoke({
    "input" : ("tell me latest news on iran and us war"
               "and then tell me current weather in new bangalore")
}
)



> Entering new AgentExecutor chain...
To answer the question, I need to first find the latest news on the situation between Iran and the US, and then provide the current weather in New Bangalore.

Action: tavily_search_results_json
Action Input: "latest news on Iran and US conflict"
[{'url': 'https://www.youtube.com/watch?v=9OToDoiWO10', 'content': "Posted: 2026-11-13\nDonald Trump says the United States no longer needs a peace deal with Iran, as the fallout of the eight-month war continues to shape American politics and regional security. After the Republicans' sweeping midterm defeat, Trump has redirected Vice President JD Vance away from negotiations with Tehran. But Iran has laid down seven conditions for ending the conflict, including sanctions relief, the removal of the US naval blockade and guarantees over the Strait of Hormuz. Meanwhile, fresh maritime attacks, Iranian claims of strikes on US assets in Bahrain and reports of Israel preparing another operation threaten to reig

In [29]:
print(response["output"])

The conflict between Iran and the US remains tense, with recent military actions including Iran targeting a US military site in Bahrain and the US shooting down Iranian drones. Diplomatic negotiations are stalled, and maritime security is affected, but oil continues to flow through the Strait of Hormuz.
